# Week 4へようこそ - LangChainとLangGraph

## Lab 1: 抽象化のレベルと、その構成要素

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">LangChainのドキュメント</h2>
            <span style="color:#00bfff;">ドキュメントは<a href="https://docs.langchain.com/oss/python/">https://docs.langchain.com/oss/python/</a>にあります。一度見てみる価値があります。ウェブ上にはまだ古い0.x時代の資料が残っているので注意してください。現代のAPIはかなり異なっています。
            </span>
        </td>
    </tr>
</table>

## 4つの抽象化レベル

LangChainとLangGraphは4段階の抽象化レベルを構成しており、それぞれが前段の上に構築されています。用語が少しわかりにくいのは、「LangChain」という語がいくつかの場所に登場するためです。

| レイヤー | パッケージ | 得られるもの | 自分が制御するもの |
|---|---|---|---|
| 1. 構成要素（Building blocks） | `langchain-core` + `langchain-openai` | chat models、`@tool`デコレータ、messages、structured output | 手作業によるtool loopを含む、すべて |
| 2. オーケストレーション | `langgraph` | state、memory、checkpointingを備えたステップのグラフ | 制御フロー（グラフを自分で設計する） |
| 3. Agent | `langchain`（`create_agent`） | あらかじめ用意された標準のagent loop | modelとtools、promptだけ |
| 4. ハーネス | `deepagents`（`create_deep_agent`） | planning、sub-agents、filesystemを備えた、あらかじめ方針の定まったハーネス | 自分の意図（intent） |


## 今週の進め方:

DAY 1（今日）: 構成要素（building blocks）  
DAY 2: LangGraph  
DAY 3: LangChain create_agent  
DAY 4: Deep Agents  
DAY 5: Sidekickプロジェクト  

## 今日はレイヤー1: 構成要素（building blocks）です

これはLangChainが始まった原点であり、LiteLLMと似たような抽象化レイヤーですが、もっと重量級のバージョンです。

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI  # メインモデルをGeminiに切り替えるため
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

### 最初のモデル呼び出し

`ChatOpenAI`はOpenAI呼び出しを包む抽象化であり、他のLLMプロバイダーにも似たようなパッケージがあります。

プロンプトを渡して`invoke`を呼び出します。これはLangChainにおける重要なメソッドです。

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-flash-latest")

message = "In 1 sentence, what does it mean for an AI Agent to be autonomous"

reply = llm.invoke(message)

print(reply.content)

### ストリーミング

トークンごとにリアルタイムで受け取りたい場合は、`invoke`を`stream`に置き換えて、chunkに対してループ処理を行います

In [ ]:
for chunk in llm.stream("Tell me a two line poem about autonomous agents"):
    print(chunk.content, end="", flush=True)

### OpenAI互換のプロバイダーなら何でも

これまでと同様に、ChatOpenAIオブジェクトを使ってOpenAI互換のエンドポイントを利用できます

In [ ]:
openrouter_llm = ChatOpenAI(
    model="anthropic/claude-haiku-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

reply = openrouter_llm.invoke("In one sentence, what is LangChain?")
print(reply.content)

### Messages

LangChainには、SystemMessage、HumanMessage、AIMessageといった抽象化が用意されていますが、これまで通りdictのリストを使うこともできます。

In [ ]:
messages = [
    SystemMessage("You are a terse assistant who answers in exactly five words."),
    HumanMessage("What is the capital of France?"),
]

print(llm.invoke(messages).content)

# まったく同じ呼び出しを、すでに知っている形式である素のdictを使って行う
messages_as_dicts = [
    {"role": "system", "content": "You are a terse assistant who answers in exactly five words."},
    {"role": "user", "content": "What is the capital of France?"},
]
print(llm.invoke(messages_as_dicts).content)

### `@tool`デコレータを使ったツール

ツールとは、モデルが呼び出せるPython関数のことです。現代的な作り方は`@tool`デコレータを使う方法です。docstringはモデルが読む説明文になり、型ヒントは引数のスキーマになります。OpenAI Agents SDKの`@function_tool`と同じ考え方です。

これは、LangChainの以前のバージョンにあった古い`Tool(...)`ラッパーに代わるものです。

In [ ]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print("name:", get_share_price.name)
print("description:", get_share_price.description)
print("args:", get_share_price.args)
print("called directly:", get_share_price.invoke({"symbol": "AAPL"}))

### モデルにツールを渡す

これは少し扱いにくいやり方です。Day 3でcreate_agentの機能を使うことで、この点は改善されます。

今のところは、Week 1でやったのと同じように、自分でループを書く必要があります。

最初のステップは、`bind_tools`を使ってツールをモデルに紐付けることです。これでinvokeを呼び出すと、モデルは回答ではなく、ツールを実行してほしいという要求を返してくることがあります。その要求は`.tool_calls`に現れます。

In [ ]:
llm_with_tools = llm.bind_tools([get_share_price])

response = llm_with_tools.invoke("What is the share price of Amazon?")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)

### tool loopを自分の手で実行する

さて、Week 1のときとかなり似た、簡単なループを書く必要があります

In [ ]:
# 会話を開始し、モデルからのツール要求を履歴に残しておく
conversation = [HumanMessage("What is the share price of Amazon?")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# 要求された各ツールを実行し、その結果をToolMessageとして追加する
for call in ai_message.tool_calls:
    if call["name"] == "get_share_price":
        result = get_share_price.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

# モデルがツールの結果を見られるようになったので、再度invokeする
final = llm_with_tools.invoke(conversation)
print(final.content)

### Structured output

OpenAI Agents SDKと同様に、モデルにPydanticのサブクラスで応答するよう要求できます

In [ ]:
class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

structured_llm = llm.with_structured_output(Company)

company = structured_llm.invoke("Tell me about Amazon the technology company")
print(company)
print("Just the ticker:", company.ticker)

## これがレイヤー1です

LiteLLMをもっと豊かに、もっと踏み込んだものにしたようなイメージです。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">演習</h2>
            <span style="color:#ff7800;">自分で2つ目のツールを書いてみましょう。例えば、ある都市の架空の天気予報を調べるようなものです。両方のツールをモデルに紐付け、両方が必要になる質問をして、モデルが最終的な答えを出すまで、tool loopを手動で実行してみてください。OpenAI Agents SDKやCrewAIよりも、地道で骨の折れる作業に感じられるはずです。
            それはDay 3で改善されます！
            </span>
        </td>
    </tr>
</table>